In [1]:
import mlflow
import mlflow.sklearn 
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np 
import re
import string 
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import os


c:\Users\singh\Desktop\mlops-mini\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import dagshub
dagshub.init(repo_owner='singhsumitt05', repo_name='mlops-mini', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/singhsumitt05/mlops-mini.mlflow')
mlflow.set_experiment('exp_2_bow_vs_tfidf')


Accessing as singhsumitt05

Initialized MLflow to track repo "singhsumitt05/mlops-mini"

Repository singhsumitt05/mlops-mini initialized!

2026/08/08 20:34:01 INFO mlflow.tracking.fluent: Experiment with name 'exp_2_bow_vs_tfidf' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5c2a41e3b8b5476795886ccf1895898a', creation_time=1786201442143, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786201442143, lifecycle_stage='active', name='exp_2_bow_vs_tfidf', tags={}, trace_location=None, workspace='default'>

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/refs/heads/main/tweet_emotions.csv')
df.drop(columns=['tweet_id'])

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...
...,...,...
39995,neutral,@JohnLloydTaylor
39996,love,Happy Mothers Day All my love
39997,love,Happy Mother's Day to all the mommies out ther...
39998,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEE...


In [4]:

import nltk 
import sys

try:
    nltk.download('wordnet', quiet=True)
    nltk.download('stopwords', quiet=True)
except Exception as e:
    print(f"Failed to download NLTK assets automatically: {e}")



def lemmatization(text: str) -> str:
    try:
        lemmatizer = WordNetLemmatizer()
        text_list = text.split()
        return " ".join([lemmatizer.lemmatize(y) for y in text_list])
    except Exception as e:
        print(f"Error in lemmatization: {e}")
        raise


def remove_stop_words(text: str) -> str:
    try:
        stop_words = set(stopwords.words("english"))
        text_list = [i for i in str(text).split() if i not in stop_words]
        return " ".join(text_list)
    except Exception as e:
        print(f"Error in remove_stop_words: {e}")
        raise

def removing_numbers(text: str) -> str:
    try:

        text_str =''.join([i for i in text if not i.isdigit()])
        return text_str
    except Exception as e:
        print(f"Error in removing_numbers: {e}")
        raise



def lower_case(text):
    
    try: 

        text_list = text.split()
        text_list =[y.lower() for y in text_list]
        return " " .join(text_list)
    except Exception as e:
        print(f"Error in lower_case: {e}")
        raise

def removing_punctuations(text):
    try: 
        ## Remove punctuations
        # FIX: Appended 'r' to strings to permanently fix the invalid escape sequence warnings
        text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,،-./:;<=>؟?@[\]^_`{|}~"""), ' ', text)
        text = text.replace('؛',"", )

        ## remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        text =  " ".join(text.split())
        return text.strip()
    except Exception as e:
        print(f"Error in removing_punctuations: {e}")
        raise



def removing_urls(text: str) -> str:
    try:
        url_pattern = re.compile(r'https?://\S+|www\.\S+')
        return url_pattern.sub(r'', str(text))
    except Exception as e:
        print(f"Error in removing_urls: {e}")
        raise


def remove_small_sentences(df):
    try:
        for i in range(len(df)):
            if len(df.text.iloc[i].split()) < 3:
                df.text.iloc[i] = np.nan
    except Exception as e:
        print(f"Error in remove_small_sentences: {e}")
        raise

def normalize_text(df: pd.DataFrame) -> pd.DataFrame:
    try:
        print("Starting text normalization pipeline...")
        df_copy = df.copy()
        
        if 'content' not in df_copy.columns:
            raise KeyError("The dataset is missing the required target text column: 'content'")
            
        # Standard Pandas explicit mapping without using chain indexing methods
        df_copy['content'] = df_copy['content'].apply(lower_case)
        df_copy['content'] = df_copy['content'].apply(remove_stop_words)
        df_copy['content'] = df_copy['content'].apply(removing_numbers)
        df_copy['content'] = df_copy['content'].apply(removing_punctuations)
        df_copy['content'] = df_copy['content'].apply(removing_urls)
        df_copy['content'] = df_copy['content'].apply(lemmatization)
        
        return df_copy
    except KeyError as e:
        print(f"Schema Error: {e}")
        raise
    except Exception as e:
        print(f"Failed during batch text preprocessing: {e}")
        raise






def main() -> None:
    try:
        print("Data Preprocessing Stage Initialized.")
        

        df_processed = normalize_text(df)
        

        
        print("Data Preprocessing Stage Completed Successfully.")
        return df_processed

    except Exception as e:
        print(f"Data preprocessing pipeline halted: {e}")
        sys.exit(1)


<>:57: SyntaxWarning: invalid escape sequence '\]'
<>:57: SyntaxWarning: invalid escape sequence '\]'
C:\Users\singh\AppData\Local\Temp\ipykernel_15840\753071338.py:57: SyntaxWarning: invalid escape sequence '\]'
  text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,،-./:;<=>؟?@[\]^_`{|}~"""), ' ', text)


In [5]:
df_processed = main()


Data Preprocessing Stage Initialized.
Starting text normalization pipeline...
Data Preprocessing Stage Completed Successfully.


In [6]:
x = df_processed['sentiment'].isin(['happiness' , 'sadness'])
df = df_processed[x]
df

,tweet_id,sentiment,content
1,1956967666,sadness,layin n bed headache ughhhh waitin call
2,1956967696,sadness,funeral ceremony gloomy friday
6,1956968487,sadness,sleep im not thinking old friend want married ...
8,1956969035,sadness,charviray charlene love miss
9,1956969172,sadness,kelcouch sorry least friday
...,...,...,...
39986,1753905153,happiness,going watch boy striped pj s hope cry
39987,1753918809,happiness,gave bike thorough wash degrease grease it thi...
39988,1753918818,happiness,amazing time last night mcfly incredible
39994,1753918900,happiness,succesfully following tayla


In [7]:
df['sentiment'] = df['sentiment'].replace({'happiness': 1, 'sadness':0})
df

C:\Users\singh\AppData\Local\Temp\ipykernel_15840\3609113855.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'happiness': 1, 'sadness':0})
C:\Users\singh\AppData\Local\Temp\ipykernel_15840\3609113855.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['sentiment'].replace({'happiness': 1, 'sadness':0})


,tweet_id,sentiment,content
1,1956967666,0,layin n bed headache ughhhh waitin call
2,1956967696,0,funeral ceremony gloomy friday
6,1956968487,0,sleep im not thinking old friend want married ...
8,1956969035,0,charviray charlene love miss
9,1956969172,0,kelcouch sorry least friday
...,...,...,...
39986,1753905153,1,going watch boy striped pj s hope cry
39987,1753918809,1,gave bike thorough wash degrease grease it thi...
39988,1753918818,1,amazing time last night mcfly incredible
39994,1753918900,1,succesfully following tayla


In [8]:
vectorizer = {
    'BoW' : CountVectorizer(),
        'Tfidf' : TfidfVectorizer()
}

algorithms = {
    'LogisticRegression' : LogisticRegression(),
    'MultinomialNB' : MultinomialNB(),
    'RandomForestClassifier' : RandomForestClassifier(),
    'XGBClassifier' : XGBClassifier(),
    'GradientBoostingClassifier' : GradientBoostingClassifier()
}

#all baselined

#start parent run 
with mlflow.start_run(run_name = 'All Experiments') as parent_run:
    #loop through all the algorithms and feature extraction method 
    for algo_name , algo in algorithms.items():
        for vec_name, vec in vectorizer.items():
            with mlflow.start_run(run_name = f"{algo_name} with {vec_name}", nested=True) as child_run:
                
                X_train_text, X_test_text, y_train, y_test = train_test_split(
                    df['content'], df['sentiment'], test_size=0.2, random_state=42
                )
                X_train = vec.fit_transform(X_train_text)
                X_test = vec.transform(X_test_text)

                mlflow.log_param('vectorizer', vec_name)
                mlflow.log_param('algorithm', algo_name)
                mlflow.log_param('test_size', 0.2)

                #model Training 
                model = algo
                model.fit(X_train, y_train)

                #log model parameter
                if algo_name == 'LogisticRegression':
                    mlflow.log_param('C', model.C) 
                    #C denotes regularization 
                    #strong regularization : less overfitting (smaller value of C) genralizes well 
                    #week regularization : overfitting (large value of C) memorizes word by word
                elif algo_name == 'MultinomialNB':
                    mlflow.log_param('alpha', model.alpha)
                elif algo_name == 'XGBClassifier':
                    mlflow.log_param('n_estimators', model.n_estimators)
                    mlflow.log_param('learning_rate', model.learning_rate)
                elif algo_name == 'RandomForestClassifier':
                    mlflow.log_param('n_estimators', model.n_estimators)
                    mlflow.log_param('max_depth', model.max_depth)
                elif algo_name == 'GradientBoostingClassifier':
                    mlflow.log_param('n_estimators', model.n_estimators)
                    mlflow.log_param('max_depth', model.max_depth)
                    mlflow.log_param('learning_rate', model.learning_rate)

                #model evaluation 
                y_pred = model.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)

                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", precision)
                mlflow.log_metric("recall", recall)
                mlflow.log_metric("f1_score", f1)

                #log model 
                if algo_name == 'XGBClassifier':
                    mlflow.xgboost.log_model(model, 'model')
                else:
                    mlflow.sklearn.log_model(model, 'model')

                #save and log notebook

                notebook_path = "exp_2_bow_vs_tfidf.ipynb"
                mlflow.log_artifact(notebook_path)

                #exp_2_bow_vs_tfidf.ipynb is presumably the notebook this code is running in. Adding --execute tells nbconvert to actually re-run every cell in that notebook — including the mlflow.start_run(run_name='All Experiments') parent loop itself. That means:

                #Every one of your 10 child runs will spawn a fresh full execution of the entire notebook, which itself loops over all 10 algorithm/vectorizer combos and logs 10 more nested runs — each of which spawns another --execute, recursively.
                #Best case, this multiplies your runtime by an enormous, unpredictable factor (potentially runs for hours/forever, or until you hit memory/kernel limits).
                #Worst case, you get overlapping kernel processes fighting over the same notebook file (--inplace writes back to disk), corrupting exp_2_bow_vs_tfidf.ipynb or causing kernel errors, since nbconvert executes with a new kernel process, not the one you're currently in.


                print(f"Algorithm: {algo_name}, Feature Engineering: {vec_name}")
                print(f"Accuracy: {accuracy}")
                print(f"Precision: {precision}")
                print(f"Recall: {recall}")
                print(f"F1 Score: {f1}")
                    
                





    


2026/08/08 20:34:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Algorithm: LogisticRegression, Feature Engineering: BoW
Accuracy: 0.7937349397590362
Precision: 0.7846750727449079
Recall: 0.7970443349753694
F1 Score: 0.7908113391984359
🏃 View run LogisticRegression with BoW at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1/runs/2d052249e0034474b4cb1c9b152484af
🧪 View experiment at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1


2026/08/08 20:35:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Algorithm: LogisticRegression, Feature Engineering: Tfidf
Accuracy: 0.7956626506024096
Precision: 0.7800947867298578
Recall: 0.8108374384236453
F1 Score: 0.7951690821256039
🏃 View run LogisticRegression with Tfidf at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1/runs/4d45c20a8e4a4ba18811e72727ad9029
🧪 View experiment at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1


2026/08/08 20:36:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

Algorithm: XGBClassifier, Feature Engineering: Tfidf
Accuracy: 0.7619277108433735
Precision: 0.7232219365895458
Recall: 0.8315270935960591
F1 Score: 0.773602199816682
🏃 View run XGBClassifier with Tfidf at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1/runs/43cb6b0cf80b4cd4b9c45514d194be8d
🧪 View experiment at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1


2026/08/08 20:43:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Algorithm: GradientBoostingClassifier, Feature Engineering: BoW
Accuracy: 0.727710843373494
Precision: 0.8008021390374331
Recall: 0.5901477832512315
F1 Score: 0.6795235394214407
🏃 View run GradientBoostingClassifier with BoW at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1/runs/8d0e2a3bdb834dfca93c6238c6914a8e
🧪 View experiment at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1


2026/08/08 20:43:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Algorithm: GradientBoostingClassifier, Feature Engineering: Tfidf
Accuracy: 0.7228915662650602
Precision: 0.8021978021978022
Recall: 0.5753694581280788
F1 Score: 0.6701090074584051
🏃 View run GradientBoostingClassifier with Tfidf at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1/runs/7c67c02d438b4f3f93cccc88b825ba7a
🧪 View experiment at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1
🏃 View run All Experiments at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1/runs/7c7549f0b175431ea137c6ade2fb618a
🧪 View experiment at: https://dagshub.com/singhsumitt05/mlops-mini.mlflow/#/experiments/1
